# qreg_outlier

> 对应代码：`EconometricsCode/code_in_notes/Chap.4/qreg_outlier.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.4`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.4")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们生成 50 个观测的模拟数据：x 与误差 u 均为标准正态，真实模型为 y = 1 + x + u，即真实的条件均值与条件中位数都是 1+x。

In [ ]:
%%stata
clear
set more off
set obs 50
set seed 505
// generate vars
gen x=rnormal()
gen u=rnormal()
// y
gen y=1+x+u

按 x 排序后把第一个观测的 y 人为改成 30，制造一个明显的离群值（outlier），用来演示 OLS 与分位数回归在稳健性上的差异。

In [ ]:
%%stata
// 产生一个outlier
sort x
replace y=30 if _n==1

先后做三个回归并保存拟合值：含离群值的 OLS；剔除离群值后的 OLS（作为"干净数据"的基准）；以及中位数回归 qreg。OLS 最小化平方损失，对离群值非常敏感；而中位数回归最小化绝对偏差，只关心残差的符号与排序，单个离群值几乎不影响估计——这正是分位数回归稳健性的来源。

In [ ]:
%%stata
// qreg
reg y x
predict p_y_reg
label variable p_y_reg "OLS"
reg y x if _n~=1
predict p_y_reg_no_outlier
label variable p_y_reg_no_outlier "OLS, 排除异常值"
qreg y x , q(0.50)
predict p_y_qreg
label variable p_y_qreg "分位数回归"

把散点与三条拟合线画在一起：可以看到含离群值的 OLS 线被明显拉向离群点，而中位数回归线与剔除离群值后的 OLS 线几乎重合。

In [ ]:
%%stata
// graph
twoway (scatter y x) (line p_y_reg  x) (line p_y_reg_no_outlier  x) (line p_y_qreg x)